<a href="https://colab.research.google.com/github/RobJavVar/DataSciencePsychNeuro/blob/master/ExerciseSubmissions/10_mixed-effects-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 10: Mixed effects

This homework assignment is designed to give you practice fitting and interpreting mixed effects models.

We will be using the **LexicalData.csv** and **Items.csv** files from the *Homework/lexDat* folder in the class GitHub repository again.

This data is a subset of the [English Lexicon Project database](https://elexicon.wustl.edu/). It provides the reaction times (in milliseconds) of many subjects as they are presented with letter strings and asked to decide, as quickly and as accurately as possible, whether the letter string is a word or not. The **Items.csv** provides characteristics of the words used, namely frequency (how common is this word?) and length (how many letters?). Unlike in the previous homework, there isn't any missing data in the **LexicalData.csv** file.

*Data courtesy of Balota, D.A., Yap, M.J., Cortese, M.J., Hutchison, K.A., Kessler, B., Loftis, B., Neely, J.H., Nelson, D.L., Simpson, G.B., & Treiman, R. (2007). The English Lexicon Project. Behavior Research Methods, 39, 445-459.*

---
## 1. Loading and formatting the data (1 point)

Load in data from the **LexicalData.csv** and **Items.csv** files. As in the previous homeworks, remove the commas from the reaction times and convert them from strings to numbers. Use `left_join` to add word characteristics `Length` and `Log_Freq_Hal` from **Items** to **LexicalData**.

*Note: the `Freq_HAL` variable in **Items.csv** has a similar formatting issue, using string values with commas. We're not going to worry about fixing this since we're only using `Log_Freq_HAL`, which is the natural log transformation of `Freq_HAL`, in this homework.*

In [9]:
# WRITE YOUR CODE HERE
library(tidyverse)

lexdata <- read_csv("/Users/Ali/Documents/GitHub/DataSciencePsychNeuro/Exercise datasets/lexDat/LexicalData_withIncorrect.csv")
items <- read_csv("/Users/Ali/Documents/GitHub/DataSciencePsychNeuro/Exercise datasets/lexDat/Items.csv")

lexdata <- lexdata %>%
  mutate(D_RT = as.numeric(gsub(",", "", D_RT)))

items_short <- items %>% select(Word, Length, Log_Freq_HAL)

lexdata <- lexdata %>%
  left_join(items_short, by = c("D_Word" = "Word"))

head(lexdata)

New names:
• `` -> `...1`
Rows: 74869 Columns: 9
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): D_Word
dbl (6): ...1, Sub_ID, Trial, Type, D_Zscore, Correct
num (1): D_RT
lgl (1): Outlier

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 30959 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (1): Word
dbl (3): Occurrences, Length, Log_Freq_HAL
num (1): Freq_HAL

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


...1,Sub_ID,Trial,Type,D_RT,D_Word,Outlier,D_Zscore,Correct,Length,Log_Freq_HAL
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<lgl>,<dbl>,<dbl>,<dbl>,<dbl>
1,157,1,1,710,browse,FALSE,-0.437,1,6,8.856
2,67,1,1,1094,refrigerant,FALSE,0.825,1,11,4.644
3,120,1,1,587,gaining,FALSE,-0.645,1,7,8.304
4,21,1,1,984,cheerless,FALSE,0.025,1,9,2.639
5,236,1,1,577,pattered,FALSE,-0.763,1,8,1.386
6,236,2,1,715,conjures,FALSE,-0.364,1,8,5.268


---
## 2. Model fitting (4 points)

First, fit a linear model with `Log_Freq_HAL` and `Length` as predictors, and `D_RT` as the output. Include an interaction term. Use `summary()` to look at the model output.

In [10]:
# WRITE YOUR CODE HERE

model1 <- lm(D_RT ~ Log_Freq_HAL*Length, data = lexdata)
summary(model1)




Call:
lm(formula = D_RT ~ Log_Freq_HAL * Length, data = lexdata)

Residuals:
    Min      1Q  Median      3Q     Max 
-1128.3  -217.6   -94.0    94.2  3317.2 

Coefficients:
                    Estimate Std. Error t value Pr(>|t|)    
(Intercept)         650.3764    14.3247  45.403  < 2e-16 ***
Log_Freq_HAL        -10.0802     1.9643  -5.132 2.88e-07 ***
Length               45.5806     1.5992  28.503  < 2e-16 ***
Log_Freq_HAL:Length  -2.6346     0.2345 -11.236  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

Residual standard error: 384.2 on 70585 degrees of freedom
  (4280 observations deleted due to missingness)
Multiple R-squared:  0.08867,	Adjusted R-squared:  0.08863 
F-statistic:  2289 on 3 and 70585 DF,  p-value: < 2.2e-16


Now, install `lme4` using `install.packages()` and then load the library.

In [13]:
# WRITE YOUR CODE HERE
install.packages("lme4")
library(lme4)



The downloaded binary packages are in
	/var/folders/xn/6zv05bqd6pg_55q7h9xf15h00000gn/T//RtmpNCdX9m/downloaded_packages


Warning message:
“package ‘lme4’ was built under R version 4.5.2”
Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack




Now fit a mixed effects model that includes the same predictors as the linear model above, as well as random intercepts for `Sub_ID` (i.e., cases where subject ID shifts the RT mean). Use `summary()` to look at the model output.

In [14]:
# WRITE YOUR CODE HERE

modelmlm <- lmer(D_RT ~ Length*Log_Freq_HAL + (1|Sub_ID), data = lexdata)
summary(modelmlm)

Linear mixed model fit by REML ['lmerMod']
Formula: D_RT ~ Length * Log_Freq_HAL + (1 | Sub_ID)
   Data: lexdata

REML criterion at convergence: 1012299

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-4.2579 -0.5401 -0.1555  0.2986 11.0922 

Random effects:
 Groups   Name        Variance Std.Dev.
 Sub_ID   (Intercept) 51061    226.0   
 Residual             97009    311.5   
Number of obs: 70589, groups:  Sub_ID, 299

Fixed effects:
                    Estimate Std. Error t value
(Intercept)         652.6262    17.4987  37.296
Length               45.7392     1.2990  35.211
Log_Freq_HAL        -11.0297     1.5960  -6.911
Length:Log_Freq_HAL  -2.5742     0.1905 -13.514

Correlation of Fixed Effects:
            (Intr) Length L_F_HA
Length      -0.635              
Log_Frq_HAL -0.621  0.913       
Lng:L_F_HAL  0.561 -0.919 -0.943

---
## 3. Model assessment (4 points)

Compare the three t-values for the fixed effects and the mixed effects models. How do they differ, and why?

> *Write your response here*
> The magnitude of the t-values are generally increased in the mixed effects model except for the intercept which is smaller. These t-values differ because the mixed effects model accounts for variability within subjects rather than assuming that each observation is independent like the linear model. Accounting for within-subject variability reduces some of the error that comes from the fixed effect and produces more accurate t-values.

Use the Aikeke Information Criterion (AIC) to compare these two models. Which one is better?

In [16]:
# WRITE YOUR CODE HERE
AIC(model1, modelmlm)


,df,AIC
,<dbl>,<dbl>
model1,5,1040519
modelmlm,6,1012311


> *Write your response here*
> According to the AIC (smaller is better model performance), the mixed effects model that includes a random intercept of subject ID is better than the linear model without the random intercept. 

---
##  4. Reflection (1 point)

What other random effects could be controlled for in this data set?

> *Write your response here*
> Could include a random slope of subject ID and word length/frequency if you think that some subjects may differ in their sensitivity to word length or frequency. Maybe also a random intercept of Word if we think that the effect of word length of frequency varies by word with some being disproportionately slower/faster than others.

**DUE:** 5pm EST, March 5, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *Someone's Name*